# DSDN: full reproduction on Google Colab (T4)
Runtime → Change runtime type → **T4 GPU**. Results are written to `OUT`; every training run resumes after a disconnect (rerun the same cell).

In [ ]:
# Setup
!pip install -q timm
import os
from google.colab import drive; drive.mount('/content/drive')
REPO  = '/content/drive/MyDrive/DSDN'                     # clone of this repository
DATA  = f'{REPO}/data/Pomegranate Diseases Dataset'      # Mendeley dataset, five class folders
CACHE_DRIVE = f'{REPO}/cache256'; CACHE = '/content/cache256'
OUT   = f'{REPO}/out'
os.chdir(REPO)
def sh(cmd): get_ipython().system(cmd)
os.makedirs(f'{OUT}/splits', exist_ok=True); sh(f'cp -n splits/*.json {OUT}/splits/')   # released splits

In [ ]:
sh(f'python dsdn.py cache --data "{DATA}" --cache {CACHE} --cache_drive {CACHE_DRIVE}')

## Dataset audit and fruit-grouped split (optional: the released `splits/` already contain the result)

In [ ]:
!pip install -q imagehash easyocr
sh('python dataset_audit.py')
sh('python group_split.py --audit audit/audit_per_image.csv --out audit_split')   # identical to splits/split_grouped.json

## Pilot check (about 5–20 min; all lines should read PASS)

In [ ]:
sh(f'python dsdn.py pilot --data "{DATA}" --cache {CACHE} --out {OUT} --split {OUT}/splits/split_random.json')

## Training (29 runs; about 18 min each on a T4)

In [ ]:
R = f'{OUT}/splits/split_random.json'; G = f'{OUT}/splits/split_grouped.json'
plan  = [(R, 'full', s) for s in (42, 123, 456)]
plan += [(R, c, 42) for c in ('no_disentangle', 'no_recon', 'no_ortho', 'no_healthy', 'matched_ctrl', 'plain_effb0')]
plan += [(G, c, 42) for c in ('full', 'matched_ctrl', 'plain_effb0')]
plan += [(f'{OUT}/splits/split_random_cv{f}.json', 'full', 42) for f in range(5)]
plan += [(R, c, 42) for c in ('bl_mobilenetv3_l', 'bl_efficientnet_b0', 'bl_efficientnet_b3', 'bl_densenet121',
                              'bl_resnet50', 'bl_resnet101', 'bl_convnext_t', 'bl_swin_t', 'bl_vit_b16')]
plan += [(R, c, 42) for c in ('sens_a0.3', 'sens_a0.7', 'sens_b0.05')]
for i, (sp, cfg, seed) in enumerate(plan, 1):
    name = f"{os.path.basename(sp)[:-5]}__{cfg}__s{seed}"
    if os.path.exists(f'{OUT}/runs/{name}/metrics.json'):
        print(f'[{i}/{len(plan)}] done: {name}'); continue
    print(f'[{i}/{len(plan)}] RUNNING {name}')
    sh(f'python dsdn.py train --cache {CACHE} --out {OUT} --split {sp} --config {cfg} --seed {seed}')

## Analyses, tables and figures

In [ ]:
sh(f'python dsdn.py probe     --cache {CACHE} --out {OUT}')
sh(f'python dsdn.py scs       --cache {CACHE} --out {OUT} --split_stem split_random')
sh(f'python dsdn.py fig9      --cache {CACHE} --out {OUT} --split_stem split_random')
sh(f'python dsdn.py farms     --data "{DATA}" --out {OUT}')
sh(f'python dsdn.py dupcheck  --out {OUT} --audit_csv audit/audit_per_image.csv --groups_csv splits/fruit_groups.csv')
sh(f'python dsdn.py stats     --out {OUT}')
sh(f'python dsdn.py benchmark --out {OUT} --bench_runs 300')
sh(f'python dsdn.py figures   --out {OUT}')
sh(f'python dsdn.py fig2      --out {OUT}')
sh(f'python dsdn.py summarize --out {OUT}')